# Question 1: Fitting a normal distribution by maximum likelihood

Ten repeated measurements of the same quantity are modeled as independent draws from a
normal distribution with unknown mean `mu` ($\mu$) and unknown variance `sigma2`
($\sigma^2$).

You will write the log-likelihood $\ell(\mu, \sigma^2)$ as a function of both parameters,
find the pair that maximizes it, and measure how precise our estimate of the mean is.

Save your own copy first (File > Save a copy in Drive). Then paste the INPUTS block from the
course page over the INPUTS cell below, replacing everything in that cell.

Run the cells in order and fill in every TODO. Use only `numpy` and `matplotlib`.

In [ ]:
# ===== INPUTS =====
# Replace this whole cell with the INPUTS block from the course page.
SAMPLE = None
MU_CHECK = None
SIGMA2_CHECK = None
# ==================

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

if SAMPLE is None or MU_CHECK is None or SIGMA2_CHECK is None:
    raise ValueError("Paste the INPUTS block from the course page over the INPUTS cell, "
                     "run that cell, then run this one again.")

DATA = np.asarray(SAMPLE, dtype=float)
n = len(DATA)          # sample size
print(f"n = {n}")
print(f"DATA = {DATA}")


def is_single_number(value):
    """Given. True when value is one finite number, as each reported value must be."""
    try:
        return np.ndim(value) == 0 and bool(np.isfinite(float(value)))
    except (TypeError, ValueError):
        return False

### TODO 1: the log-likelihood

Complete `loglik` so that it returns the normal log-likelihood of the whole sample at the
parameter pair (`mu`, `sigma2`) $= (\mu, \sigma^2)$:

$$\ell(\mu, \sigma^2) \; = \; \sum_{i=1}^{n} \log f(x_i \mid \mu, \sigma^2),
\qquad
f(x \mid \mu, \sigma^2) \; = \; \frac{1}{\sqrt{2\pi\sigma^2}} \exp\!\left(-\frac{(x - \mu)^2}{2\sigma^2}\right)$$

Note that `sigma2` is the variance $\sigma^2$, not the standard deviation $\sigma$.
Compute whatever you need from `DATA` inside the function. Hint: think sufficient statistics.

Write it for scalar `mu` and `sigma2`; the contour plot below evaluates it point by point.
R1 on the course page is this function at the pair (`MU_CHECK`, `SIGMA2_CHECK`) from your
INPUTS block. The cell right after your function prints it.

In [ ]:
def loglik(mu, sigma2, DATA):
    """Normal log-likelihood of the sample at (mu, sigma2)."""
    # TODO 1: return the log-likelihood, computing what you need from DATA
    n = len(DATA)
    return ...

In [ ]:
# Given. Prints R1 as soon as TODO 1 is done; it does not depend on the later steps.
try:
    r1 = loglik(MU_CHECK, SIGMA2_CHECK, DATA)
except Exception as err:
    r1 = err
if is_single_number(r1):
    r1_line = f"R1. loglik at (MU_CHECK, SIGMA2_CHECK) (4 dec):  {float(r1):.4f}"
else:
    r1_line = ("R1 not available: loglik should return one number, the log-likelihood "
               f"of the whole sample. It returned {r1!r:.80}")
print(r1_line)

### TODO 2: the maximizers

Complete `mle` so that it returns the pair (`mu_hat`, `sigma2_hat`)
$= (\hat\mu, \hat\sigma^2)$ that maximizes the log-likelihood, in closed form, from `DATA`
alone:

$$(\hat\mu, \hat\sigma^2) \; = \; \arg\max_{\mu, \, \sigma^2} \; \ell(\mu, \sigma^2)$$

Be careful with the variance: the maximum-likelihood estimate $\hat\sigma^2$ divides by $n$,
so it is not the usual sample variance with divisor $n - 1$.

In [ ]:
def mle(DATA):
    """The maximum-likelihood estimates (mu_hat, sigma2_hat)."""
    # TODO 2: return the closed-form maximizers as a tuple
    return ...

In [ ]:
# Given. Prints R2 as soon as TODO 2 is done; it does not depend on TODO 1.
try:
    mu_hat, sigma2_hat = mle(DATA)
except Exception:
    mu_hat = sigma2_hat = None
if is_single_number(mu_hat) and is_single_number(sigma2_hat):
    mu_hat, sigma2_hat = float(mu_hat), float(sigma2_hat)
    print(f"mu_hat = {mu_hat:.4f}")
    r2_line = f"R2. sigma2_hat (4 dec):  {sigma2_hat:.4f}"
else:
    mu_hat = sigma2_hat = None
    r2_line = "R2 not available: mle should return two numbers, (mu_hat, sigma2_hat)."
print(r2_line)

### How precisely the mean is determined

Treating $\sigma^2$ as a constant, the curvature of the log-likelihood in the $\mu$
direction is minus its second derivative:

$$ C \; = \; -\left. \frac{\partial^{2} \ell}{\partial \mu^{2}} \right|_{\mu = \hat\mu, \; \sigma^{2} = \hat\sigma^{2}} $$

A larger $C$ means a narrower peak. The standard error of `mu_hat` ($\hat\mu$) is one over
the square root of that curvature:

$$ \mathrm{se}(\hat\mu) \; = \; \frac{1}{\sqrt{C}} $$

`se_of_mean` is given. It measures $C$ numerically from your `loglik`, by comparing the
log-likelihood at $\hat\mu$ with its values a small step to either side.

In [ ]:
def se_of_mean(DATA):
    """Given. Standard error of mu_hat, from the curvature of your log-likelihood in mu.

    Returns None when the curvature is not a single positive number.
    """
    mu_hat, sigma2_hat = mle(DATA)
    h = 0.01 * np.sqrt(sigma2_hat)      # a small step in mu
    try:
        with np.errstate(all="ignore"):
            C = -(loglik(mu_hat + h, sigma2_hat, DATA) - 2 * loglik(mu_hat, sigma2_hat, DATA)
                  + loglik(mu_hat - h, sigma2_hat, DATA)) / h**2
    except Exception:
        return None
    if not is_single_number(C) or C <= 0:
        return None
    return 1 / np.sqrt(C)


se_mu = se_of_mean(DATA) if sigma2_hat is not None else None
if se_mu is None:
    print("Skipping the standard error and the plot: the curvature of your loglik at the "
          "maximum is not a single positive number, so check loglik in TODO 1 (and mle in "
          "TODO 2). R2 does not depend on this step.")
else:
    print(f"se of mean = {se_mu:.4f}")

### Compare with the usual sample variance

The cell below prints your maximum-likelihood variance $\hat\sigma^2$ next to
`np.var(DATA, ddof=1)`, which divides by $n - 1$ instead of $n$. They are not the same
quantity. NumPy's default, `np.var(DATA)`, divides by $n$ and does match $\hat\sigma^2$.

This is a self-check to help you arrive at the right answer, not a reported value. If the two come out equal, revisit TODO 2.

In [ ]:
if sigma2_hat is None:
    print("Complete TODO 2 first.")
else:
    print(f"MLE variance,        divides by n:     {sigma2_hat:.4f}")
    print(f"np.var(DATA, ddof=1), divides by n - 1: {np.var(DATA, ddof=1):.4f}")
    print(f"ratio of the two:                       {np.var(DATA, ddof=1) / sigma2_hat:.4f}")
    print(f"n / (n - 1):                            {n / (n - 1):.4f}")

### See the peak

The log-likelihood is a 2D plot over the two parameters $(\mu, \sigma^2)$. The plot
below shows it, along with your estimates $(\hat\mu, \hat\sigma^2)$ marked and a horizontal bar
spanning one standard error on either side of $\hat\mu$.

In [ ]:
if se_mu is None:
    print("Plot skipped until the standard error above is available.")
else:
    mu_grid = np.linspace(mu_hat - 4 * se_mu, mu_hat + 4 * se_mu, 120)
    s2_grid = np.linspace(0.35 * sigma2_hat, 2.6 * sigma2_hat, 120)
    LL = np.array([[loglik(m, s2, DATA) for m in mu_grid] for s2 in s2_grid])

    plt.figure(figsize=(7, 4.5))
    plt.contourf(mu_grid, s2_grid, LL, levels=30)
    plt.colorbar(label="l(mu, sigma2)")
    plt.contour(mu_grid, s2_grid, LL, levels=12, colors="white", linewidths=0.5, alpha=0.6)
    plt.plot(mu_hat, sigma2_hat, "r*", markersize=15, label="maximum")
    plt.plot([mu_hat - se_mu, mu_hat + se_mu], [sigma2_hat, sigma2_hat],
             color="red", linewidth=2, label="one standard error in mu")
    plt.xlabel("mu")
    plt.ylabel("sigma2")
    plt.title("Log-likelihood surface, and its maximum")
    plt.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

### Report your results

R1 and R2 were printed after TODO 1 and TODO 2. The cell below collects them; enter them on
the course page.

In [ ]:
print("=" * 60)
print("QUESTION 1: REPORT VALUES")
print("=" * 60)
print(r1_line)
print(r2_line)

### Multiple-choice A

*This also appears on the course page, where you answer it. The options there may be in a different order.*

Hold $\sigma^2$ fixed at any positive value. For every data set, which objective is minimized by
the same $\mu$ that maximizes the log-likelihood $\ell(\mu, \sigma^2)$?

1. The sum of squared residuals, $\sum_i (x_i - \mu)^2$.
2. The sum of absolute residuals, $\sum_i |x_i - \mu|$.
3. The largest absolute residual, $\max_i |x_i - \mu|$.
4. The sum of fourth powers of the residuals, $\sum_i (x_i - \mu)^4$.

### Multiple-choice B

*This also appears on the course page, where you answer it. The options there may be in a different order.*

The red bar in your contour plot spans one standard error either side of $\hat\mu$. Suppose
a second set of ten measurements gave the same $\hat\mu$ but a larger $\hat\sigma^2$. Which
statement comparing the estimate from the second set of measurements to the estimate from
the first set is best supported?

1. The standard error of $\hat\mu$ would be unchanged, because it depends on how many measurements there are and not on how spread out they are.
2. $\hat\mu$ would move, because the estimate of the mean depends on the estimate of the variance.
3. The peak would be flatter in the $\mu$ direction, and the standard error of $\hat\mu$ would be larger.
4. The standard error of $\hat\mu$ would be smaller, because a larger variance spreads the likelihood over more values of $\mu$.

### Multiple-choice C

*This also appears on the course page, where you answer it. The options there may be in a different order.*

Suppose every observation were doubled, $x_i \mapsto 2 x_i$: the same measurements read off an
instrument calibrated in half-sized units. Without re-running anything, which set of statements
would hold?

1. $\hat\mu$ doubles, $\hat\sigma^2$ doubles, and the standard error of the mean doubles.
2. $\hat\mu$ doubles, $\hat\sigma^2$ is multiplied by 4, and the standard error of the mean doubles.
3. $\hat\mu$ is unchanged, $\hat\sigma^2$ is multiplied by 4, and the standard error of the mean doubles.
4. $\hat\mu$ doubles, $\hat\sigma^2$ is multiplied by 4, and the standard error of the mean is multiplied by 4.